In [ ]:
import numpy as np
import pandas as pd
import time
import datetime
import gc
import random
import re
import ast
from sklearn import preprocessing
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler,random_split
from sklearn.model_selection import train_test_split

import transformers
from transformers import BertForSequenceClassification, BertConfig,BertTokenizer,get_linear_schedule_with_warmup, AutoModelForSequenceClassification
from torch.optim import AdamW


In [ ]:
from transformers import AutoModel, AutoTokenizer
model_name = "dlicari/Italian-Legal-BERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
mlb = MultiLabelBinarizer()

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

In [ ]:
df = pd.read_csv("./data/ft_train_encoded.csv")
df = df.rename(columns={'title' :'TitleLaw', 'ministeries': 'label'})
df

In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

# The labels are already encoded in the CSV, so we need to parse them
labels_list = []
for row in df['label']:
    # Parse the string representation of the list
    ministry_list = ast.literal_eval(row)
    labels_list.append(ministry_list)

mlb = MultiLabelBinarizer()
encoded_labels = mlb.fit_transform(labels_list)
num_classes = len(mlb.classes_)
print(f"Classes: {mlb.classes_}")
print(f"Number of classes: {num_classes}")

df['labels'] = list(encoded_labels)
df

In [ ]:
titles = df.TitleLaw.values
labels = encoded_labels

### BERT Tokenizer


In [ ]:
# Load the Italian Legal BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
max_len = 0
i = 0
lab = []
inp = []
tit = []
# For every sentence...
for sent in titles:

    # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
    input_ids = tokenizer.encode(sent, add_special_tokens=True)

    if len(input_ids) <= 512:
        lab.append(labels[i])
        inp.append(input_ids)
        tit.append(titles[i])
        # Update the maximum sentence length.
        max_len = max(max_len, len(input_ids))
    i = i+1

input_ids = inp
labels = lab
titles = tit
print('Max sentence length: ', max_len)

In [ ]:
input_ids = []
attention_masks = []

# For every tweet...
for title in titles:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
                        title,                      # Sentence to encode.
                        add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                        max_length = max_len,           # Pad & truncate all sentences.
                        padding = 'max_length',
                        truncation = True,
                        return_attention_mask = True,   # Construct attn. masks.
                        return_tensors = 'pt',     # Return pytorch tensors.
                   )

    # Add the encoded sentence to the list.
    input_ids.append(encoded_dict['input_ids'])

    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels)

#### Train-validation split
80% of data is split into train and 20% to validation sets.

In [ ]:
# Combine the training inputs into a TensorDataset.
dataset = TensorDataset(input_ids, attention_masks, labels)

# Create a 90-10 train-validation split.

# Calculate the number of samples to include in each set.
train_size = int(0.8 * len(dataset))
val_size = len(dataset)  - train_size

# Divide the dataset by randomly selecting samples.
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print('{:>5,} training samples'.format(train_size))
print('{:>5,} validation samples'.format(val_size))

In [ ]:
# The DataLoader needs to know our batch size for training, so we specify it
# here. For fine-tuning BERT on a specific task, the authors recommend a batch
# size of 16 or 32.
batch_size = 32

# Create the DataLoaders for our training and validation sets.
# We'll take training samples in random order.
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

# For validation the order doesn't matter, so we'll just read them sequentially.
validation_dataloader = DataLoader(
            val_dataset, # The validation samples.
            sampler = SequentialSampler(val_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

In [ ]:
# Load AutoModelForSequenceClassification for multi-label classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    problem_type="multi_label_classification"
)

model = model.to(device)

In [ ]:
optimizer = AdamW(model.parameters(),
                  lr = 2e-5, # args.learning_rate - default is 5e-5, our notebook had 2e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8.
                )

# Fine tuning the model

In [ ]:
# Number of training epochs. The BERT authors recommend between 2 and 4.
# We chose to run for 4, but we'll see later that this may be over-fitting the
# training data.
epochs = 4

# Total number of training steps is [number of batches] x [number of epochs].
# (Note that this is not the same as the number of training samples).
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

In [ ]:
from sklearn.metrics import f1_score
import numpy as np
import torch

# Function to calculate metrics for multi-label classification
def compute_metrics(preds, labels):
    # Apply sigmoid and threshold for multi-label
    preds_sigmoid = torch.sigmoid(torch.tensor(preds)).numpy()
    preds_binary = (preds_sigmoid >= 0.5).astype(int)
    
    f1_micro = f1_score(labels, preds_binary, average='micro')
    f1_macro = f1_score(labels, preds_binary, average='macro')
    
    return f1_micro, f1_macro

In [ ]:
def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [ ]:
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)
training_stats = []
best_f1_micro = 0  # Initialize best F1 score tracking

# Measure the total training time for the whole run.
total_t0 = time.time()

# For each epoch...
for epoch_i in range(0, epochs):

    # ========================================
    #               Training
    # ========================================
    # Perform one full pass over the training set.
    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')
    # Measure how long the training epoch takes.
    t0 = time.time()
    total_train_loss = 0
    model.train()
    for step, batch in enumerate(train_dataloader):
        # Unpack this training batch from our dataloader.
        #
        # As we unpack the batch, we'll also copy each tensor to the device using the
        # `to` method.
        #
        # `batch` contains three pytorch tensors:
        #   [0]: input ids
        #   [1]: attention masks
        #   [2]: labels
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device).float()  # Convert to float for multi-label
        optimizer.zero_grad()
        output = model(b_input_ids,
                             token_type_ids=None,
                             attention_mask=b_input_mask,
                             labels=b_labels)
        loss = output.loss
        total_train_loss += loss.item()
        # Perform a backward pass to calculate the gradients.
        loss.backward()
        # Clip the norm of the gradients to 1.0.
        # This is to help prevent the "exploding gradients" problem.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        # Update parameters and take a step using the computed gradient.
        # The optimizer dictates the "update rule"--how the parameters are
        # modified based on their gradients, the learning rate, etc.
        optimizer.step()
        # Update the learning rate.
        scheduler.step()

    # Calculate the average loss over all of the batches.
    avg_train_loss = total_train_loss / len(train_dataloader)

    # Measure how long this epoch took.
    training_time = format_time(time.time() - t0)
    print("")
    print("  Average training loss: {0:.2f}".format(avg_train_loss))
    print("  Training epoch took: {:}".format(training_time))
    # ========================================
    #               Validation
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our validation set.
    print("")
    print("Running Validation...")
    t0 = time.time()
    # Put the model in evaluation mode--the dropout layers behave differently
    # during evaluation.
    model.eval()
    # Tracking variables
    total_eval_loss = 0
    nb_eval_steps = 0
    all_preds = []
    all_labels = []
    
    # Evaluate data for one epoch
    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device).float()  # Convert to float for multi-label
        # Tell pytorch not to bother with constructing the compute graph during
        # the forward pass, since this is only needed for backprop (training).
        with torch.no_grad():
            output= model(b_input_ids,
                                   token_type_ids=None,
                                   attention_mask=b_input_mask,
                                   labels=b_labels)
        loss = output.loss
        total_eval_loss += loss.item()
        # Move logits and labels to CPU if we are using GPU
        logits = output.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.detach().cpu().numpy()
        
        all_preds.extend(logits)
        all_labels.extend(label_ids)
    
    # Calculate metrics for multi-label classification
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    f1_micro, f1_macro = compute_metrics(all_preds, all_labels)
    
    print("  F1 Micro: {0:.4f}".format(f1_micro))
    print("  F1 Macro: {0:.4f}".format(f1_macro))
    
    # Calculate the average loss over all of the batches.
    avg_val_loss = total_eval_loss / len(validation_dataloader)
    # Measure how long the validation run took.
    validation_time = format_time(time.time() - t0)
    
    # Save best model based on F1 micro score
    if epoch_i == 0 or f1_micro > best_f1_micro:
        best_f1_micro = f1_micro
        torch.save(model, './bert_model')
    
    # Record all statistics from this epoch.
    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'F1 Micro': f1_micro,
            'F1 Macro': f1_macro,
            'Training Time': training_time,
            'Validation Time': validation_time
        }
    )
print("")
print("Training complete!")

print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))

## Model Saving

Save the trained model and tokenizer for future use.

In [ ]:
# Save the model, tokenizer, and label encoder
import os
import pickle

# Create directory for saving
model_dir = './saved_model'
os.makedirs(model_dir, exist_ok=True)

# Save the model and tokenizer
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

# Save the MultiLabelBinarizer for label decoding
with open(os.path.join(model_dir, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(mlb, f)

# Save training statistics
with open(os.path.join(model_dir, 'training_stats.pkl'), 'wb') as f:
    pickle.dump(training_stats, f)

print(f"Model saved to {model_dir}")
print(f"Best F1 Micro score: {best_f1_micro:.4f}")

In [ ]:
# Make predictions on Swiss titles
print("Making predictions on Swiss law titles...")
print(f"Processing {len(swiss_titles)} titles...")

# Convert to list and handle any NaN values
swiss_titles_clean = [str(title) if pd.notna(title) else "" for title in swiss_titles]

# Make predictions
predictions, probabilities = predict_ministries(
    swiss_titles_clean, 
    model, 
    tokenizer, 
    mlb, 
    device, 
    batch_size=16  # Smaller batch size to avoid memory issues
)

print(f"Predictions completed! Shape: {predictions.shape}")
print(f"Probabilities shape: {probabilities.shape}")

## Model Loading and Predictions

Load the saved model and make predictions on swiss_titles.csv

In [ ]:
# Load the saved model and components
import os
import pickle
from transformers import AutoModelForSequenceClassification, AutoTokenizer
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model_dir = './saved_model'

# Load the model and tokenizer
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_dir)
loaded_tokenizer = AutoTokenizer.from_pretrained(model_dir)

# Load the MultiLabelBinarizer
with open(os.path.join(model_dir, 'label_encoder.pkl'), 'rb') as f:
    loaded_mlb = pickle.load(f)

# Move model to device
loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Model loaded successfully!")
print(f"Number of classes: {len(loaded_mlb.classes_)}")
print(f"Classes: {loaded_mlb.classes_}")

In [ ]:
# Load swiss_titles.csv
swiss_df_full = pd.read_csv('./data/swiss_titles.csv')
print(f"Loaded {len(swiss_df_full)} Swiss law titles")

# Take only first 10,000 titles
swiss_df = swiss_df_full.head(10000)
print(f"Processing only {len(swiss_df)} titles")
print("\nFirst few rows:")
print(swiss_df.head())

# Extract titles for prediction
swiss_titles = swiss_df['title_it'].values
print(f"\nNumber of titles to predict: {len(swiss_titles)}")

In [ ]:
def predict_ministries(texts, model, tokenizer, mlb, device, batch_size=32, max_length=512):
    """
    Predict ministries for a list of texts
    """
    model.eval()
    all_predictions = []
    all_probabilities = []
    
    # Process texts in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize batch
        encoded = tokenizer(
            batch_texts,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        # Move to device
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        
        # Make predictions
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Apply sigmoid to get probabilities
            probs = torch.sigmoid(logits).cpu().numpy()
            
            # Apply threshold to get binary predictions
            preds = (probs >= 0.5).astype(int)
            
            all_predictions.extend(preds)
            all_probabilities.extend(probs)
    
    return np.array(all_predictions), np.array(all_probabilities)

print("Prediction function defined!")

In [ ]:
# Make predictions on Swiss titles
print("Making predictions on Swiss law titles...")
print(f"Processing {len(swiss_titles)} titles...")

# Convert to list and handle any NaN values
swiss_titles_clean = [str(title) if pd.notna(title) else "" for title in swiss_titles]

# Make predictions
predictions, probabilities = predict_ministries(
    swiss_titles_clean, 
    loaded_model, 
    loaded_tokenizer, 
    loaded_mlb, 
    device, 
    batch_size=16  # Smaller batch size to avoid memory issues
)

print(f"Predictions completed! Shape: {predictions.shape}")
print(f"Probabilities shape: {probabilities.shape}")

In [ ]:
# Convert predictions back to ministry labels
predicted_ministries = mlb.inverse_transform(predictions)

# Create results dataframe
results_df = swiss_df.copy()
results_df['predicted_ministries'] = [list(ministries) for ministries in predicted_ministries]
results_df['num_predicted_ministries'] = [len(ministries) for ministries in predicted_ministries]

# Add probability scores for each ministry
for i, ministry in enumerate(mlb.classes_):
    results_df[f'prob_{ministry}'] = probabilities[:, i]

# Display some statistics
print("\nPrediction Statistics:")
print(f"Total documents: {len(results_df)}")
print(f"Documents with at least one ministry: {(results_df['num_predicted_ministries'] > 0).sum()}")
print(f"Documents with no ministries: {(results_df['num_predicted_ministries'] == 0).sum()}")
print(f"Average ministries per document: {results_df['num_predicted_ministries'].mean():.2f}")

# Show distribution of predicted ministries
print("\nMinistry prediction counts:")
ministry_counts = {}
for ministries in predicted_ministries:
    for ministry in ministries:
        ministry_counts[ministry] = ministry_counts.get(ministry, 0) + 1

for ministry, count in sorted(ministry_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{ministry}: {count}")

In [ ]:
# Display some sample predictions
print("\nSample Predictions:")
print("=" * 80)

# Randomly sample 10 predictions
sample_indices = np.random.choice(len(results_df), size=min(10, len(results_df)), replace=False)

for i in sample_indices:
    title = results_df.iloc[i]['title_it']
    predicted = results_df.iloc[i]['predicted_ministries']
    
    print(f"\nTitle: {title}")
    print(f"Predicted Ministries: {predicted}")
    
    # Show top 3 probabilities
    probs = probabilities[i]
    top_indices = np.argsort(probs)[-3:][::-1]
    print("Top 3 probabilities:")
    for idx in top_indices:
        ministry = mlb.classes_[idx]
        prob = probs[idx]
        print(f"  {ministry}: {prob:.3f}")
    print("-" * 60)

In [ ]:
# Save results to CSV
results_csv_path = './data/swiss_titles_predictions.csv'
results_df.to_csv(results_csv_path, index=False)

# Also save a simplified version with just the main results
simplified_results = results_df[['act', 'title_it', 'predicted_ministries', 'num_predicted_ministries']].copy()
simplified_csv_path = './data/swiss_titles_predictions_simple.csv'
simplified_results.to_csv(simplified_csv_path, index=False)

print(f"\nResults saved to:")
print(f"Full results: {results_csv_path}")
print(f"Simplified results: {simplified_csv_path}")

print(f"\nFiles created:")
print(f"- Model directory: {model_dir}")
print(f"- Predictions (full): {results_csv_path}")
print(f"- Predictions (simple): {simplified_csv_path}")